# 04 — Verify LiteRT/TFLite model trên local Windows

Mục tiêu của notebook này là xác nhận các file `.tflite` export từ Kaggle **thực sự chạy được trên máy local** trước khi đưa vào Android Studio.

Project hiện tại dự kiến có cấu trúc:

```text
TrafficSignAI/
├── models/
│   ├── pretrained/
│   ├── trained/
│   │   ├── yolo11n_320_best.pt
│   │   ├── yolo11n_640_best.pt
│   │   ├── yolo11s_320_best.pt
│   │   └── yolo11s_640_best.pt
│   └── deploy/
│       ├── yolo11n_320.tflite
│       ├── yolo11n_640.tflite
│       ├── yolo11s_320.tflite
│       ├── yolo11s_640.tflite
│       ├── labels.txt
│       └── model_info.txt
├── notebooks/
├── runs/
└── VR-TSD-2/
```

Notebook sẽ làm 4 việc:

1. kiểm tra file và runtime LiteRT;
2. inspect tensor contract ngay trên Windows;
3. chạy cùng một nhóm ảnh test qua cả 4 `.tflite`;
4. tùy chọn validate toàn bộ test set để so với metric `.pt`.

Không chỉnh sửa model và không train lại.


## 1. Kiểm tra môi trường

Nếu `ai_edge_litert` chưa có trong `.venv`, chạy cell cài đặt bên dưới một lần.


In [2]:
# Kiểm tra Python và package chính
import sys
import platform

print("Python:", sys.version)
print("OS:", platform.platform())

try:
    import ultralytics
    print("Ultralytics:", ultralytics.__version__)
except ImportError:
    print("Thiếu ultralytics")

try:
    import ai_edge_litert
    print("ai-edge-litert: OK")
except ImportError:
    print("ai-edge-litert: CHƯA CÀI")


Python: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
OS: Windows-10-10.0.19045-SP0
Ultralytics: 8.4.142
ai-edge-litert: OK


### Chỉ chạy cell này nếu cell trên báo `ai-edge-litert: CHƯA CÀI`

Sau khi cài xong, restart kernel rồi chạy notebook lại từ đầu.


In [3]:
# Bỏ comment dòng dưới nếu cần cài LiteRT runtime trên Windows
# %pip install -U ai-edge-litert


## 2. Xác định project root và các model

Notebook tự tìm `D:\Project\TrafficSignAI` khi chạy từ folder `notebooks`.


In [4]:
from pathlib import Path

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path(r"D:\Project\TrafficSignAI"),
]

PROJECT_ROOT = None

for candidate in candidate_roots:
    if (candidate / "VR-TSD-2").exists() and (candidate / "models").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy project TrafficSignAI.")

TRAINED_DIR = PROJECT_ROOT / "models" / "trained"
DEPLOY_DIR = PROJECT_ROOT / "models" / "deploy"
TEST_IMAGES_DIR = PROJECT_ROOT / "VR-TSD-2" / "test" / "images"
DATA_YAML = PROJECT_ROOT / "VR-TSD-2" / "data_local.yaml"

MODELS = {
    "yolo11n_320": {
        "pt": TRAINED_DIR / "yolo11n_320_best.pt",
        "tflite": DEPLOY_DIR / "yolo11n_320.tflite",
        "imgsz": 320,
    },
    "yolo11n_640": {
        "pt": TRAINED_DIR / "yolo11n_640_best.pt",
        "tflite": DEPLOY_DIR / "yolo11n_640.tflite",
        "imgsz": 640,
    },
    "yolo11s_320": {
        "pt": TRAINED_DIR / "yolo11s_320_best.pt",
        "tflite": DEPLOY_DIR / "yolo11s_320.tflite",
        "imgsz": 320,
    },
    "yolo11s_640": {
        "pt": TRAINED_DIR / "yolo11s_640_best.pt",
        "tflite": DEPLOY_DIR / "yolo11s_640.tflite",
        "imgsz": 640,
    },
}

print("Project root:", PROJECT_ROOT)
print("Test images:", TEST_IMAGES_DIR)
print()

for name, spec in MODELS.items():
    print(
        name,
        "| PT:", spec["pt"].exists(),
        "| TFLite:", spec["tflite"].exists(),
        "| imgsz:", spec["imgsz"],
    )


Project root: D:\Project\TrafficSignAI
Test images: D:\Project\TrafficSignAI\VR-TSD-2\test\images

yolo11n_320 | PT: True | TFLite: True | imgsz: 320
yolo11n_640 | PT: True | TFLite: True | imgsz: 640
yolo11s_320 | PT: True | TFLite: True | imgsz: 320
yolo11s_640 | PT: True | TFLite: True | imgsz: 640


## 3. Kiểm tra `labels.txt`

Dataset có 58 class. File `labels.txt` phải có đúng 58 dòng và đúng thứ tự class.


In [5]:
LABELS_PATH = DEPLOY_DIR / "labels.txt"

labels = [
    line.strip()
    for line in LABELS_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("labels.txt:", LABELS_PATH)
print("Số class:", len(labels))
print("10 class đầu:")

for idx, label in list(enumerate(labels))[:10]:
    print(idx, label)

assert len(labels) == 58, f"Expected 58 classes, got {len(labels)}"


labels.txt: D:\Project\TrafficSignAI\models\deploy\labels.txt
Số class: 58
10 class đầu:
0 102-cam-di-nguoc-chieu
1 103a-cam-oto
2 103b-cam-oto-re-phai
3 103c-cam-oto-re-trai
4 106-cam-oto-tai
5 107-cam-oto-khach-va-oto-tai
6 123a-cam-re-trai
7 123b-cam-re-phai
8 124a-cam-quay-dau-xe
9 124b-cam-oto-quay-dau-xe


## 4. Inspect tensor contract ngay trên Windows

Kaggle đã inspect một lần. Cell này xác nhận các file tải về vẫn được LiteRT runtime trên Windows mở bình thường.


In [6]:
from ai_edge_litert.interpreter import Interpreter

tensor_contracts = {}

for name, spec in MODELS.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    interpreter = Interpreter(model_path=str(spec["tflite"]))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    tensor_contracts[name] = {
        "input": input_details,
        "output": output_details,
    }

    for x in input_details:
        print("INPUT :", x["shape"], x["dtype"])

    for x in output_details:
        print("OUTPUT:", x["shape"], x["dtype"])



yolo11n_320
INPUT : [  1   3 320 320] <class 'numpy.float32'>
OUTPUT: [   1   62 2100] <class 'numpy.float32'>

yolo11n_640
INPUT : [  1   3 640 640] <class 'numpy.float32'>
OUTPUT: [   1   62 8400] <class 'numpy.float32'>

yolo11s_320
INPUT : [  1   3 320 320] <class 'numpy.float32'>
OUTPUT: [   1   62 2100] <class 'numpy.float32'>

yolo11s_640
INPUT : [  1   3 640 640] <class 'numpy.float32'>
OUTPUT: [   1   62 8400] <class 'numpy.float32'>


## 5. Chọn cùng một nhóm ảnh test cho cả 4 model

Dùng seed cố định để lần sau chạy lại vẫn lấy đúng các ảnh này.


In [7]:
import random

image_files = sorted([
    p for p in TEST_IMAGES_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
])

random.seed(42)
sample_count = min(8, len(image_files))
sample_images = random.sample(image_files, sample_count)

print("Số ảnh test tổng:", len(image_files))
print("Ảnh smoke test:")

for p in sample_images:
    print("-", p.name)


Số ảnh test tổng: 1151
Ảnh smoke test:
- ch0_20250430105000_20250430105300_f00288_jpg.rf.ff911426361b58657908d0bc8641a681.jpg
- 4_mp4-0020_jpg.rf.d83658bfc8daacc6130291fd9389449e.jpg
- Eddy2612-online-video-cutter_com-_mp4-0049_jpg.rf.4e4cdb7b8067b123efdef52d0ea245e1.jpg
- ch0_20250517120708_20250517121208_f03978_jpg.rf.a7db81db2c1cf46e5e6b87bba22c8372.jpg
- ch0_20250504163332_20250504163632_f04020_jpg.rf.5fea7fffc3a908d01593f97b003a8404.jpg
- ch0_20250430105600_20250430105900_f00324_jpg.rf.7770b43363a67dd5e5c03f0954cf687f.jpg
- ch0_20250430104659_20250430104959_f02394_jpg.rf.b066fb535c3445bf2158d28431a01b03.jpg
- NO20250710-172749-000012F_f03055_jpg.rf.f2a3a7c462bb3ff5e0896afd3f26c397.jpg


## 6. Smoke inference cả 4 `.tflite`

Tất cả model xử lý **cùng một nhóm ảnh**.

Kết quả prediction được lưu vào:

`runs/litert_smoke/<model_name>/`

Mục tiêu ở bước này chưa phải benchmark tốc độ; chỉ kiểm tra model có inference ra detection hợp lý hay không.


In [8]:
from ultralytics import YOLO

SMOKE_DIR = PROJECT_ROOT / "runs" / "litert_smoke"
smoke_summary = []

for name, spec in MODELS.items():
    print("\n" + "=" * 70)
    print("SMOKE:", name)
    print("=" * 70)

    model = YOLO(str(spec["tflite"]))

    results = model.predict(
        source=[str(p) for p in sample_images],
        imgsz=spec["imgsz"],
        conf=0.25,
        device="cpu",
        save=True,
        verbose=False,
        project=str(SMOKE_DIR),
        name=name,
        exist_ok=True,
    )

    total_detections = sum(len(r.boxes) for r in results)

    smoke_summary.append({
        "model": name,
        "imgsz": spec["imgsz"],
        "images": len(results),
        "detections": total_detections,
    })

    print("Images:", len(results))
    print("Total detections:", total_detections)
    print("Saved to:", SMOKE_DIR / name)



SMOKE: yolo11n_320
Loading D:\Project\TrafficSignAI\models\deploy\yolo11n_320.tflite for LiteRT inference...


ValueError: Cannot set tensor: Dimension mismatch. Got 8 but expected 1 for dimension 0 of input 0.

## 7. Xem bảng smoke-test

Nếu cả 4 model đều chạy xong mà không exception, đây là bằng chứng mạnh hơn nhiều so với chỉ có file `.tflite`.


In [ ]:
import pandas as pd

smoke_df = pd.DataFrame(smoke_summary)
display(smoke_df)


## 8. So trực quan `.pt` và `.tflite` của model chính `YOLO11n-640`

Đây là model ưu tiên đầu tiên cho Android.

Ta chạy cùng 8 ảnh bằng:
- `yolo11n_640_best.pt`
- `yolo11n_640.tflite`

Sau đó mở hai folder prediction để so bounding box, class và confidence.


In [ ]:
MAIN_NAME = "yolo11n_640"
main_spec = MODELS[MAIN_NAME]

pt_model = YOLO(str(main_spec["pt"]))
tflite_model = YOLO(str(main_spec["tflite"]))

compare_dir = PROJECT_ROOT / "runs" / "pt_vs_litert_smoke"

_ = pt_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    verbose=False,
    project=str(compare_dir),
    name="pt",
    exist_ok=True,
)

_ = tflite_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=640,
    conf=0.25,
    device="cpu",
    save=True,
    verbose=False,
    project=str(compare_dir),
    name="tflite",
    exist_ok=True,
)

print("PT predictions     :", compare_dir / "pt")
print("TFLite predictions :", compare_dir / "tflite")


## 9. Full validation — chạy sau khi smoke test ổn

Bước này dùng toàn bộ test set để kiểm tra accuracy sau conversion.

Mặc định chỉ validate `YOLO11n-640` vì đây là candidate Android chính. Sau khi model này ổn, có thể thêm ba model còn lại vào `MODELS_TO_VALIDATE`.

> LiteRT validation chạy CPU nên có thể chậm hơn `.pt` chạy RTX 4060.


In [ ]:
# Chọn model cần full validation
MODELS_TO_VALIDATE = [
    "yolo11n_640",
    # "yolo11n_320",
    # "yolo11s_320",
    # "yolo11s_640",
]

validation_rows = []

for name in MODELS_TO_VALIDATE:
    spec = MODELS[name]

    print("\n" + "=" * 70)
    print("FULL VALIDATION:", name)
    print("=" * 70)

    model = YOLO(str(spec["tflite"]))

    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=spec["imgsz"],
        batch=16,
        device="cpu",
        workers=4,
        plots=True,
        project=str(PROJECT_ROOT / "runs" / "litert_validation"),
        name=name,
        exist_ok=True,
    )

    validation_rows.append({
        "experiment": name,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)


## 10. So với metric `.pt` đã có

Notebook thử đọc `overnight_summary.csv` từ kết quả train/evaluate hôm trước và ghép với LiteRT metrics.

Mục tiêu:
- metric `.tflite` phải gần `.pt`;
- nếu tụt mạnh thì chưa đưa model lên Android.


In [ ]:
summary_candidates = list(
    (PROJECT_ROOT / "runs").rglob("overnight_summary.csv")
)

if summary_candidates:
    summary_path = summary_candidates[0]
    pt_summary = pd.read_csv(summary_path)

    print("PT summary:", summary_path)

    wanted_cols = [
        "experiment",
        "test_precision",
        "test_recall",
        "test_mAP50",
        "test_mAP50_95",
    ]

    if all(col in pt_summary.columns for col in wanted_cols):
        comparison = validation_df.merge(
            pt_summary[wanted_cols],
            on="experiment",
            how="left",
        )

        comparison["delta_precision"] = (
            comparison["precision"] - comparison["test_precision"]
        )
        comparison["delta_recall"] = (
            comparison["recall"] - comparison["test_recall"]
        )
        comparison["delta_mAP50"] = (
            comparison["mAP50"] - comparison["test_mAP50"]
        )
        comparison["delta_mAP50_95"] = (
            comparison["mAP50_95"] - comparison["test_mAP50_95"]
        )

        display(comparison)
    else:
        print("overnight_summary.csv không có đúng bộ cột mong đợi.")
else:
    print("Không tìm thấy overnight_summary.csv. Có thể so metric thủ công.")


## Sau notebook này

Nếu `YOLO11n-640.tflite`:
- load được bằng LiteRT trên Windows;
- inference ảnh thật bình thường;
- prediction gần `.pt`;
- full-test metrics gần `.pt`;

thì model được xem là đủ tin cậy để chuyển sang Android.

Bước tiếp theo khi đó là:

```text
TrafficSignApp
→ app/src/main/assets/
→ yolo11n_640.tflite
→ labels.txt
→ LiteRTDetector.kt
→ CameraX frame preprocessing
→ inference
→ decode output [1, 62, 8400]
→ NMS
→ bounding box thật
```
